# TakeMeter — Colab pipeline

This notebook runs the full TakeMeter pipeline on Google Colab: install dependencies, upload or mount data, preprocess, quick-train a DistilBERT model, run prediction, and evaluate. Set Runtime ▶ Change runtime type ▶ GPU before running.


## 1) Colab 環境の確認

次のセルで Python バージョン、CUDA と GPU 情報、該当ライブラリ（torch）の有無を確認します。

In [1]:
# Colab environment info
import sys
import torch
import platform

print('Python', sys.version)
print('Platform', platform.platform())
try:
    print('torch', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA device name:', torch.cuda.get_device_name(0))
except Exception as e:
    print('torch not available or error:', e)


Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform Linux-6.6.122+-x86_64-with-glibc2.35
torch 2.11.0+cu128
CUDA available: True
CUDA device name: Tesla T4


## 2) 必要ライブラリのインストールとインポート

次のセルで Colab 上での依存をインストールします（GPUランタイムを使う前提）。失敗した場合はメッセージに従って CPU 向けの代替コマンドを試してください。

In [2]:
# Install dependencies (GPU-preferred). If this fails, try the CPU line below.
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets scikit-learn matplotlib pandas

# Fallback (CPU-only) - uncomment if needed
#!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.3 MB/s eta 0:00:00a 0:00:01


In [3]:
# Imports
import os
import sys
import subprocess
import json
import time
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
print('imports ok')


imports ok


## 3) 入力データの準備

以下のセルで、(A) ローカルからアップロードする方法、または (B) Google Drive をマウントしてファイルを使う方法を示します。`data/` フォルダの中に `takemeter_labeled.csv` を置いてください。開発時は `takemeter_synthetic_200.csv` を使っても構いません。

In [6]:
# Option A: upload file from local
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

# If you uploaded a CSV, move it into data/
import os
os.makedirs('data', exist_ok=True)
for fn in uploaded:
    # if filename seems like a CSV, save
    if fn.lower().endswith('.csv'):
        os.rename(fn, os.path.join('data', fn))
        print('Moved', fn, '-> data/')


KeyboardInterrupt: 

In [7]:
# Option B: mount Google Drive (use if you placed repo or data in Drive)
from google.colab import drive
drive.mount('/content/drive')
# Example: copy from drive
# !cp -r /content/drive/MyDrive/your_folder/* .
print('Mounted Drive. If your repo is in Drive, copy files into the working dir.')


Mounted at /content/drive
Mounted Drive. If your repo is in Drive, copy files into the working dir.


## 4) 主要処理の実行（前処理 → 訓練 → 推論 → 評価）

以下のセルは、リポジトリにあるスクリプトをそのまま呼び出す想定です。もしリポジトリが Colab セッションにない場合は、Drive からコピーするか、ファイルをアップロードしてください。

In [8]:
# 4a) 前処理
!python3 scripts/preprocess_and_split.py

# show resulting files
!ls -l data | sed -n '1,200p'


python3: can't open file '/content/scripts/preprocess_and_split.py': [Errno 2] No such file or directory
ls: cannot access 'data': No such file or directory


In [9]:
# 4b) 訓練（短時間のスモークテスト）
# quick_train.py は内部で torch.device を自動検出します
!python3 scripts/quick_train.py


python3: can't open file '/content/scripts/quick_train.py': [Errno 2] No such file or directory


In [ ]:
# 4c) 推論
!python3 scripts/predict.py

# show predictions head
!head -n 10 outputs/predictions.csv


In [ ]:
# 4d) 評価
!python3 scripts/evaluate.py --preds outputs/predictions.csv --gold data/val.csv --out results/metrics.json

# show metrics
!cat results/metrics.json
from IPython.display import Image, display
if os.path.exists('results/confusion_matrix.png'):
    display(Image('results/confusion_matrix.png'))


## 5) 実行手順（上から順に実行してください）

1. Runtime > Change runtime type > GPU を選択してランタイムを再起動する。 
2. すべてのセルを上から順に実行する。 
3. `data/` に `takemeter_labeled.csv` を配置するか、アップロード/Driveマウントを使う。
4. 訓練が終わったら `results/metrics.json` と `results/confusion_matrix.png` をダウンロードして提出物に含める。

## 6) 結果の保存とダウンロード

以下のセルで `outputs/` と `results/` を zip にまとめ、ダウンロード可能にします。Drive に保存することも可能です。

In [ ]:
# zip outputs and results
!zip -r takeMeter_results.zip outputs results data || true
from google.colab import files
files.download('takeMeter_results.zip')


## 3.5) リポジトリ配置の確認

`/content` 直下に `scripts/` がないと `python3 scripts/...` は失敗します。以下のセルのどちらかで、リポジトリ本体を Colab に置いてから次へ進んでください。

In [ ]:
# Option 1: if the repo is already on Google Drive, copy it here.
# Replace the path below with your actual Drive location.
# Example:
# !cp -r /content/drive/MyDrive/ai201-project3-takemeter /content/ai201-project3-takemeter
# %cd /content/ai201-project3-takemeter

# Option 2: if your repo is on GitHub, clone it here.
# Replace USER/REPO with the real GitHub path.
# !git clone https://github.com/USER/REPO.git /content/ai201-project3-takemeter
# %cd /content/ai201-project3-takemeter

import os
print('cwd:', os.getcwd())
print('scripts exists:', os.path.exists('scripts'))
print('preprocess exists:', os.path.exists('scripts/preprocess_and_split.py'))
print('quick_train exists:', os.path.exists('scripts/quick_train.py'))


# If the checks above print `scripts exists: False`, stop here and copy/clone the repo first.
# After the repo is in place, rerun the preprocessing/training cells below.
